Reading From Bronze Table



In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
df.display()

**Data Tarnsformation**

# **Trimming**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

**Normalization**

In [0]:
# df = (
#     df
#     .withColumn(
#         "cst_marital_status",
#         F.when(F.upper(F.col("cst_marital_status")) == 'S' , "Single")
#          .when(F.upper(F.col("cst_marital_status")) == "M",'Married')
#          .otherwise("n/a")
#     )
#     .withColumn(
#         "cst_gndr",
#         F.when(F.upper(F.col('cst_gndr')) == 'F','Female')
#          .when(F.upper(F.col('cst_gndr')) == 'M','Male')
#          .otherwise('n/a')

#     )
# )

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == 'S' , "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M",'Married')
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col('cst_gndr')) == 'F','Female')
         .when(F.upper(F.col('cst_gndr')) == 'M','Male')
         .otherwise("n/a")
    )
    
)
df.display()

# **Renaming the columns**

In [0]:
RENAME_MAP = {
    'cst_id':'customer_id',
    'cst_key':'customer_key',
    'cst_firstname':'first_name',
    'cst_lastname':'last_name',
    'cst_marital_status':'marital_status',
    'cst_gndr':'gender',
    'cst_create_date':'created_date'
}

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)
df.display()

# **# _Write in to silver table_**

In [0]:
df = (
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_customers")
)